# Step 3: 도구 실행 오케스트레이션 — 직렬/병렬 자동 결정

## 📋 학습 목표

- Claude Code가 **여러 도구 호출을 병렬/직렬로 자동 분류**하는 방법을 이해합니다
- `partitionToolCalls()` 의 **배치 분류 알고리즘**을 배웁니다
- Python `asyncio.gather()`로 **실제 병렬 실행**을 구현하고 성능 차이를 측정합니다

---

## 🔍 Claude Code 소스 분석

### 3-1. 문제: LLM이 한 턴에 여러 도구를 호출한다

LLM은 한 번의 응답에서 여러 도구를 동시에 요청할 수 있습니다:

```
사용자: "src/main.ts, src/Tool.ts, src/query.ts 세 파일을 읽어줘"

LLM 응답:
  tool_use: Read({file_path: "src/main.ts"})
  tool_use: Read({file_path: "src/Tool.ts"})
  tool_use: Read({file_path: "src/query.ts"})
```

이 3개의 Read를 순차 실행하면 3배의 시간이 걸립니다. 하지만 파일 읽기는 서로 간섭하지 않으므로 동시에 실행해도 안전합니다.

반면, `Write({path: "a.txt"})` 다음에 `Read({path: "a.txt"})`가 오면 순서가 중요합니다.

### 3-2. runTools() — 오케스트레이션의 핵심 (toolOrchestration.ts:19-82)

```typescript
// src/services/tools/toolOrchestration.ts:19-82

export async function* runTools(toolUseMessages, ...) {
  let currentContext = toolUseContext

  // 1. 도구 호출을 배치로 분류
  for (const { isConcurrencySafe, blocks } of partitionToolCalls(toolUseMessages, currentContext)) {

    if (isConcurrencySafe) {
      // 2a. 병렬 안전 배치 → 동시 실행
      for await (const update of runToolsConcurrently(blocks, ...)) {
        yield { message: update.message, newContext: currentContext }
      }
    } else {
      // 2b. 비안전 배치 → 순차 실행
      for await (const update of runToolsSerially(blocks, ...)) {
        if (update.newContext) currentContext = update.newContext
        yield { message: update.message, newContext: currentContext }
      }
    }
  }
}
```

### 3-3. partitionToolCalls() — 배치 분류 (toolOrchestration.ts:91-116)

```typescript
// 연속된 병렬안전 도구를 하나의 배치로 합치는 알고리즘
function partitionToolCalls(toolUseMessages, toolUseContext): Batch[] {
  return toolUseMessages.reduce((acc, toolUse) => {
    const tool = findToolByName(tools, toolUse.name)
    const isConcurrencySafe = tool?.isConcurrencySafe(toolUse.input) ?? false

    // 이전 배치가 병렬이고, 현재도 병렬이면 → 같은 배치에 합침
    if (isConcurrencySafe && acc.at(-1)?.isConcurrencySafe) {
      acc.at(-1).blocks.push(toolUse)
    } else {
      // 새 배치 생성
      acc.push({ isConcurrencySafe, blocks: [toolUse] })
    }
    return acc
  }, [])
}
```

**예시:** `[Read, Read, Bash, Read, Read, Write]` 호출이 오면:

| 배치 | 도구 | 실행 방식 |
|------|------|-----------|
| 1 | Read, Read | 병렬 |
| 2 | Bash | 직렬 |
| 3 | Read, Read | 병렬 |
| 4 | Write | 직렬 |

배치 간은 순차 실행되지만, 배치 내의 병렬 도구는 동시에 실행됩니다.

---

## 🏗️ Python으로 구현하기

`mini_claude/orchestrator.py`에 구현된 오케스트레이터를 확인합니다.

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.llm_client import ToolCall
from mini_claude.tool_registry import ToolRegistry
from mini_claude.tools.bash_tool import BashTool
from mini_claude.tools.file_read_tool import FileReadTool
from mini_claude.tools.file_write_tool import FileWriteTool
from mini_claude.orchestrator import partition_tool_calls, run_tools, Batch

# 레지스트리 셋업
registry = ToolRegistry()
registry.register(BashTool)
registry.register(FileReadTool)
registry.register(FileWriteTool)

# 배치 분류 테스트: [Read, Read, Bash, Read, Write]
test_calls = [
    ToolCall(id="1", name="Read", input={"file_path": "/tmp/a.txt"}),
    ToolCall(id="2", name="Read", input={"file_path": "/tmp/b.txt"}),
    ToolCall(id="3", name="Bash", input={"command": "echo hello"}),
    ToolCall(id="4", name="Read", input={"file_path": "/tmp/c.txt"}),
    ToolCall(id="5", name="Write", input={"file_path": "/tmp/d.txt", "content": "test"}),
]

batches = partition_tool_calls(test_calls, registry)
print("배치 분류 결과:")
for i, batch in enumerate(batches):
    mode = "병렬" if batch.is_concurrent else "직렬"
    names = [tc.name for tc in batch.calls]
    print(f"  배치 {i+1}: {names} → {mode}")

### 성능 비교: 병렬 vs 직렬

느린 도구 (sleep이 포함된 Bash)를 여러 번 호출하여 병렬/직렬 성능 차이를 측정합니다.

In [ ]:
import time
import asyncio
from mini_claude.orchestrator import execute_single_tool, run_tools

# 0.5초 걸리는 Bash 명령 3개를 병렬로 실행
slow_calls = [
    ToolCall(id=f"slow_{i}", name="Bash", input={"command": "sleep 0.5 && echo done"})
    for i in range(3)
]

# 직렬 실행 (Bash는 concurrency_safe=False이므로 기본적으로 직렬)
start = time.monotonic()
serial_results = await run_tools(slow_calls, registry)
serial_time = (time.monotonic() - start) * 1000

# 강제 병렬 실행으로 비교 (asyncio.gather 직접 사용)
start = time.monotonic()
parallel_results = await asyncio.gather(
    *[execute_single_tool(tc, registry) for tc in slow_calls]
)
parallel_time = (time.monotonic() - start) * 1000

print(f"직렬 실행 (3 × Bash): {serial_time:.0f}ms")
print(f"병렬 실행 (3 × Bash): {parallel_time:.0f}ms")
print(f"속도 향상: {serial_time / parallel_time:.1f}x")
print()
print("⚠️ Bash는 concurrency_safe=False이므로 오케스트레이터가 자동으로 직렬 실행합니다.")
print("   Read는 concurrency_safe=True이므로 자동으로 병렬 실행됩니다.")

---

## 💡 핵심 설계 원칙 정리

### 1. 자동 분류 기반 오케스트레이션
도구 자체가 `is_concurrency_safe()` 를 선언하므로, 오케스트레이터는 도구의 구체적 동작을 몰라도 안전하게 병렬/직렬을 결정할 수 있습니다.

### 2. 연속 병렬 합치기 (배치 최적화)
`[Read, Read, Bash, Read, Read]` → `[[Read,Read](병렬), [Bash](직렬), [Read,Read](병렬)]`. 연속된 안전 도구를 하나의 배치로 합쳐서 `gather()` 한 번으로 처리합니다.

### 3. 최대 동시 실행 제한
Claude Code는 `MAX_TOOL_USE_CONCURRENCY=10` 으로 동시 실행 수를 제한합니다. 리소스 고갈을 방지하는 안전 장치입니다.

---

## ➡️ 다음 Step 미리보기

**Step 4**에서는 MCP(Model Context Protocol)를 통해 **외부 도구 서버**를 연결합니다. 빌트인 도구(Bash, Read, Write)와 외부 MCP 도구가 동일한 `Tool` 인터페이스로 통합되어 에이전트가 사용할 수 있게 됩니다.